## 1. BigQuery 연결

In [2]:
from google.cloud import bigquery

PROJECT_ID = "sns-analysis-prj"
DATA_SET = "sns_analysis"
TABLE_ID = f"{PROJECT_ID}.{DATA_SET}.hackle_properties"

client = bigquery.Client(project=PROJECT_ID)

print(TABLE_ID)

sns-analysis-prj.sns_analysis.hackle_properties


/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


## 2. 삭제 대상 확인

회원 단위 분석을 위해 `user_id`가 비어 있거나 숫자로만 구성되지 않은 행을 제거합니다.  
실제 삭제 전에 각 유형의 행 수를 확인합니다.

In [3]:
sql = f"""
    SELECT
        COUNT(*) AS total_rows,

        COUNTIF(
            user_id IS NULL
        ) AS null_user_rows,

        COUNTIF(
            user_id IS NOT NULL
            AND TRIM(user_id) = ''
        ) AS blank_user_rows,

        COUNTIF(
            TRIM(user_id) != ''
            AND NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS non_numeric_user_rows,

        COUNTIF(
            user_id IS NULL
            OR TRIM(user_id) = ''
            OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS delete_rows

    FROM `{TABLE_ID}`
"""

check_before = client.query(sql).to_dataframe()
display(check_before)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,total_rows,null_user_rows,blank_user_rows,non_numeric_user_rows,delete_rows
0,525350,0,82255,109004,191259


## 3. 결측·비숫자 user_id 제거

아래 DML 셀을 실행하면 BigQuery의 `hackle_properties` 원본 테이블에서 해당 행이 실제로 삭제됩니다.

In [4]:
sql = f"""
    DELETE FROM `{TABLE_ID}`
    WHERE user_id IS NULL
       OR TRIM(user_id) = ''
       OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
"""

query_job = client.query(sql)
query_job.result()

print(
    f"삭제 완료! 삭제된 행 수: "
    f"{query_job.num_dml_affected_rows:,}건"
)

삭제 완료! 삭제된 행 수: 191,259건


## 4. 전처리 결과 확인

삭제 후 남은 행 수와 비정상 `user_id` 존재 여부를 확인합니다.

In [5]:
sql = f"""
    SELECT
        COUNT(*) AS remaining_rows,

        COUNT(DISTINCT TRIM(user_id)) AS remaining_users,

        COUNTIF(
            user_id IS NULL
            OR TRIM(user_id) = ''
            OR NOT REGEXP_CONTAINS(TRIM(user_id), r'^[0-9]+$')
        ) AS invalid_user_rows

    FROM `{TABLE_ID}`
"""

check_after = client.query(sql).to_dataframe()
display(check_after)

/Users/apple/DA15_Part4/sns_service_analysis/.venv312/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,remaining_rows,remaining_users,invalid_user_rows
0,334091,230853,0


## 전처리 결과

- 전체 525,350건 중 `user_id`가 빈 문자열인 82,255건을 제거했습니다.
- 숫자로만 구성되지 않은 `user_id` 109,004건은 익명 또는 이벤트 추적용 식별자로 판단하여 회원 단위 분석에서 제외했습니다.
- 총 191,259건을 제거했으며, 전처리 후 334,091건이 남았습니다.
- 전처리 후 남은 고유 숫자형 회원 ID는 230,853개입니다.
- 세션 및 기기 ID는 회원 식별에 사용하지 않으므로 별도로 수정하거나 제거하지 않았습니다.